# A3.2 · Sandboxed execution

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.1 · Default-deny on the tool call](https://spbreed.github.io/cyber-commons/lessons/A3.1.html)**.

| | |
|---|---|
| Tools used | Kubernetes NetworkPolicy, seccomp, Terraform, gVisor |

## What this lesson is

**What it covers.** Run the same code inside and outside the sandbox and enumerate what each could reach.

**Why a security engineer needs it.** Model-authored code inherits the runtime's reach, including any credential mounted into the environment. The control it builds is: execution in an isolate with no ambient credentials, a bounded filesystem and no default network.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

"It runs in a sandbox" is not a control until a manifest says what the sandbox contains. A container with the host network and a mounted socket is a deployment convenience wearing the word — and a namespace with no default-deny NetworkPolicy is default-allow, however many narrow policies you wrote.

> **At CyberTravels.** The Coding Agent runs generated code, and the File System Agent runs on Alex's laptop. “It runs in a sandbox” is not a control until somebody says whether that sandbox can see `~/.aws` and the HR folder. R6.

## 2 · The framework

```
   "it runs in a sandbox"  ->  contains what, exactly?

   filesystem   only the workspace, or the host's?
   network      none, allowlist, or the host's network namespace?
   sockets      is the container runtime socket mounted in?
   syscalls     full kernel surface, or a filtered one?
   credentials  is a production token mounted inside it?

   and the network answer is two Kubernetes objects, not one:

     NetworkPolicy default-deny-all   podSelector: {}     <- makes
       policyTypes: [Ingress, Egress]                        every pod
                                                             RESTRICTED
     NetworkPolicy workflow-agent-egress                   <- adds back
       egress: bookings-db:5432, kube-dns:53                  two hops

   policies are additive allow-lists. delete the first and the
   second stops being a restriction, silently, with nothing failing
```

**Mitigates: T11 Malicious Code Execution · T2 Tool Misuse.**

For an agent that runs code, the sandbox **is** the security boundary. Not the
prompt, not the code review, not the model's training. The question is never
"is there a sandbox" but "what does this one actually contain".

A1.8's lesson was that reach is a property of the environment, not of intent —
the benign task touched a private key because `open()` sees what the process
sees. So the control is to change what the process sees.

Four dimensions, and the fourth is the one teams get wrong:

**Filesystem.** A bounded working directory. Not the home directory, which holds
`.ssh`, `.aws` and `.config`.

**Process and syscall.** No spawning, no ptrace, resource ceilings so a runaway
loop is contained rather than fatal.

**Network.** No egress by default. Not "restricted" — none, and then an
explicit allow per destination the workload genuinely needs.

**Credentials.** The one people miss: **a sandbox with production credentials
mounted in it is not a sandbox.** Isolation of the filesystem is irrelevant if
the environment holds a token that reaches production over a network the
sandbox does permit. The strongest boundary in the world does not help when the
keys are inside it.

### Say it in the manifest, or you have not said it

"No egress by default" is a claim about a cluster, and a claim about a cluster
is worth what its manifest says. In Kubernetes that is two objects and they are
both required, because a `NetworkPolicy` is an **additive allow-list**: pods
that no policy selects are unrestricted, and policies never deny — they only
add permitted traffic to a pod that some policy has already made restricted.

So the pattern is: one policy that selects every pod in the namespace and
permits nothing, then one narrow policy per destination the agent needs. Delete
the first and the second stops being a restriction at all, silently, with every
pod still running and every test still green.

The same two-object shape appears in every cloud. On AWS a security group with
no egress rules plus one rule per endpoint, and an IAM policy whose `Condition`
binds the role to the workload identity rather than to a subnet. On GCP a
hierarchical firewall policy with a low-priority `deny` on `0.0.0.0/0` and
higher-priority `allow` rules. Different nouns, identical structure: deny
everything by construction, then name what is permitted, one destination at a
time.

> **What this control closes.**
>
> Changes what the executing process can **reach**, which is the only variable A1.8 turned on. A sandbox holding production credentials contains nothing that matters.

## 3 · The network dimension, as a manifest

Two objects, both required. The first makes every pod in the namespace
restricted and permits nothing; the second adds back exactly one destination.

```yaml
# 1. Default-deny. Selects EVERY pod in the namespace, permits no egress and
#    no ingress. Without this object the policy below is not a restriction —
#    it is an allowance on a pod that was already unrestricted.
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: default-deny-all
  namespace: prod-agents
spec:
  podSelector: {}                 # every pod
  policyTypes: [Ingress, Egress]  # both, or egress stays wide open
---
# 2. One destination, for one agent, selected by the same service account that
#    carries its SPIFFE ID. DNS is separate and explicit: without port 53 the
#    agent cannot resolve anything, which is a correct default and a confusing
#    first afternoon.
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: workflow-agent-egress
  namespace: prod-agents
spec:
  podSelector:
    matchLabels:
      app.kubernetes.io/name: workflow-agent   # sa/workflow-agent
  policyTypes: [Egress]
  egress:
    - to:
        - namespaceSelector:
            matchLabels: {kubernetes.io/metadata.name: prod-data}
          podSelector:
            matchLabels: {app: bookings-db}
      ports:
        - {protocol: TCP, port: 5432}
    - to:
        - namespaceSelector:
            matchLabels: {kubernetes.io/metadata.name: kube-system}
          podSelector:
            matchLabels: {k8s-app: kube-dns}
      ports:
        - {protocol: UDP, port: 53}
```

Note what is **not** in the allow-list, and what that costs an attacker:
`169.254.169.254` — the cloud metadata service, which hands out the node's
credentials to anything that can reach it — is unreachable because nothing
named it, not because anybody thought of it. That is the property a deny-list
can never have.

The pod itself carries the other three dimensions:

```yaml
spec:
  serviceAccountName: workflow-agent
  automountServiceAccountToken: false   # no ambient cluster credential
  securityContext:
    runAsNonRoot: true
    runAsUser: 10001
    seccompProfile: {type: RuntimeDefault}
  containers:
    - name: agent
      securityContext:
        allowPrivilegeEscalation: false
        readOnlyRootFilesystem: true
        capabilities: {drop: [ALL]}
      resources:
        limits: {cpu: "1", memory: 1Gi}
      volumeMounts:
        - {name: work, mountPath: /sandbox/work}   # the only writable path
  volumes:
    - name: work
      emptyDir: {sizeLimit: 512Mi}
```

The same shape on **AWS** — a security group whose egress rules are the whole
allow-list, and a role whose trust policy binds to the workload rather than to
the subnet:

```hcl
resource "aws_security_group" "workflow_agent" {
  name   = "workflow-agent"
  vpc_id = var.vpc_id
  # No `egress` block at all: an AWS security group with no egress rules
  # permits nothing outbound. The default SG that ships with a VPC allows
  # 0.0.0.0/0 — never attach that one.
}

resource "aws_vpc_security_group_egress_rule" "bookings_db" {
  security_group_id            = aws_security_group.workflow_agent.id
  referenced_security_group_id = aws_security_group.bookings_db.id
  ip_protocol                  = "tcp"
  from_port                    = 5432
  to_port                      = 5432
}
```

```json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Action": ["s3:GetObject"],
    "Resource": "arn:aws:s3:::cybertravels-itineraries/*",
    "Condition": {
      "StringEquals": {
        "aws:PrincipalTag/spiffe-id":
          "spiffe://cybertravels.com/ns/prod/sa/workflow-agent"
      },
      "Bool": {"aws:SecureTransport": "true"}
    }
  }]
}
```

And on **GCP**, where the deny is explicit and priority-ordered rather than
implicit:

```yaml
# gcloud compute network-firewall-policies rules create ...
- priority: 65000            # lowest priority: the floor
  direction: EGRESS
  action: deny
  match: {destIpRanges: ["0.0.0.0/0"]}
- priority: 1000             # higher priority wins
  direction: EGRESS
  action: allow
  targetSecureTags: ["tagValues/workflow-agent"]
  match:
    destIpRanges: ["10.20.0.0/24"]
    layer4Configs: [{ipProtocol: tcp, ports: ["5432"]}]
```

Three products, one structure: deny everything by construction, then name what
is permitted, one destination at a time.

## Your turn

Run `kubectl get networkpolicy -A` and look for a policy with an empty `podSelector` and `policyTypes: [Ingress, Egress]`. If there isn't one in the namespace your agents run in, every narrow policy you have written is an allowance rather than a restriction, and every pod nobody wrote a policy for has the internet.

---

**Next → [A3.3 · Egress control](https://spbreed.github.io/cyber-commons/lessons/A3.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*